## Basic analysis

In [2]:
import pandas as pd
import statsmodels.formula.api as smf

def run_logit_analysis(llm: str, sentiment_classifier: str, type: str):
    path = f'data/all_data_baisc_instruction.csv'
    df = pd.read_csv(path)
    df = df[df['model']==llm]
    df = df[df['sentiment_aliyun'].isin({'positive', 'negative', 'neutral'})]

    if type == 'in':
        df['sentiment_bin'] = df[f'sentiment_{sentiment_classifier}'].apply(lambda x: 1 if x == 'positive' else 0)
        df['group_bin'] = pd.Categorical(df['group'], categories=["they"] + [x for x in df['group'].unique() if x != "they"], ordered=True)
    if type == 'out':
        df['sentiment_bin'] = df[f'sentiment_{sentiment_classifier}'].apply(lambda x: 1 if x == 'negative' else 0)
        df['group_bin'] = pd.Categorical(df['group'], categories=["we"] + [x for x in df['group'].unique() if x != "we"], ordered=True)

    formula = 'sentiment_bin ~ C(group_bin) + TTR + TotalTokenScaled'

    model = smf.logit(formula, data=df)
    result = model.fit(disp=False)

    params = result.params.filter(like='C(group_bin)')
    coef = params.values[0]
    std = result.bse[params.index].values[0]
    pval = result.pvalues[params.index].values[0]
    return {'LLM': llm, 'SentimentClassifier': sentiment_classifier, 'Coefficient': coef, 'Std': std, 'PValue': pval, 'type': type}

def batch_logit_analysis(llm_list, sentiment_list, output_csv=''):
    results = []
    for llm in llm_list:
        for sentiment in sentiment_list:
            result = run_logit_analysis(llm, sentiment, type='in')
            results.append(result)
            print(llm+' '+sentiment+' OK')
    for llm in llm_list:
        for sentiment in sentiment_list:
            result = run_logit_analysis(llm, sentiment, type='out')
            results.append(result)
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv, index=False)
    print(f"Results saved to {output_csv}")


llm_list = ['Qwen3-8B-Base', 'Qwen-7B', 'Baichuan2-7B-Base', 'glm-4-9b-hf', 'Yi-1.5-6B', 'Qwen3-8B', 'deepseek-v3', 'ernie-4.5-turbo-128k', 'qwenplus', 'hunyuan']
sentiment_list = ['aliyun']


batch_logit_analysis(llm_list, sentiment_list, '.\\data\\logistic_result.csv')

PatsyError: Error evaluating factor: NameError: name 'TTR' is not defined
    sentiment_bin ~ C(group_bin) + TTR + TotalTokenScaled
                                   ^^^

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 读取数据
data = pd.read_csv('./data/logistic_result.csv')  # 修改为适配 Unix/Windows 的路径写法
data['Coefficient'] = np.exp(data['Coefficient'])

# 定义 p 值格式化函数
def format_pvalue(p):
    if p < 1e-4:
        return 'p<1e-4'
    else:
        return f'{p:.1e}'

# 定义绘图函数
def plot_group_heatmap(group, data):
    sub_data = data[data['type'] == group].copy()
    sub_data['Label'] = sub_data.apply(
        lambda row: f"{row['Coefficient']:.2f}\n({format_pvalue(row['PValue'])})", axis=1)

    coef_matrix = sub_data.pivot(index='LLM', columns='SentimentClassifier', values='Coefficient')
    label_matrix = sub_data.pivot(index='LLM', columns='SentimentClassifier', values='Label')

    plt.figure(figsize=(8, 4))
    sns.heatmap(coef_matrix, annot=label_matrix, fmt='', cmap='Blues', center=0, 
                linewidths=0.5, cbar_kws={'label': 'Coefficient'})
    plt.title(f'Odds Ratio Heatmap with p-Value ({group})')
    plt.tight_layout()
    # 如果需要保存
    # plt.savefig(f'./data/heat_map_{group}.pdf', format='pdf')
    plt.show()

# 对 'in' 和 'out' 分别绘图
for group in ['in', 'out']:
    plot_group_heatmap(group, data)
